In [2]:
from pathlib import Path

print("Current directory:")
print(Path.cwd())

print("\nCSV files available:")
for file in Path.cwd().rglob("*.csv"):
    print(file.resolve())

Current directory:
c:\Users\suriy\Desktop\healthcare_fraud\experiments

CSV files available:
C:\Users\suriy\Desktop\healthcare_fraud\experiments\data\X_test_62.csv
C:\Users\suriy\Desktop\healthcare_fraud\experiments\data\X_train_62.csv
C:\Users\suriy\Desktop\healthcare_fraud\experiments\data\y_test_62.csv
C:\Users\suriy\Desktop\healthcare_fraud\experiments\data\y_train_62.csv
C:\Users\suriy\Desktop\healthcare_fraud\experiments\outputs\feature_importance_300.csv
C:\Users\suriy\Desktop\healthcare_fraud\experiments\outputs\selected_62_feature_analysis.csv
C:\Users\suriy\Desktop\healthcare_fraud\experiments\outputs\selected_nonzero_features.csv
C:\Users\suriy\Desktop\healthcare_fraud\experiments\outputs\top_50_features.csv


In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print("Project root:")
print(PROJECT_ROOT)

print("\nSearching for original CSV files...")

for file in PROJECT_ROOT.rglob("*.csv"):
    print(file)

Project root:
c:\Users\suriy\Desktop\healthcare_fraud

Searching for original CSV files...
c:\Users\suriy\Desktop\healthcare_fraud\X_test.csv
c:\Users\suriy\Desktop\healthcare_fraud\X_train.csv
c:\Users\suriy\Desktop\healthcare_fraud\X_val.csv
c:\Users\suriy\Desktop\healthcare_fraud\y_test.csv
c:\Users\suriy\Desktop\healthcare_fraud\y_train.csv
c:\Users\suriy\Desktop\healthcare_fraud\y_val.csv
c:\Users\suriy\Desktop\healthcare_fraud\outputs\claim_predictions.csv
c:\Users\suriy\Desktop\healthcare_fraud\outputs\provider_predictions.csv
c:\Users\suriy\Desktop\healthcare_fraud\.venv\Lib\site-packages\matplotlib\mpl-data\sample_data\data_x_x2_x3.csv
c:\Users\suriy\Desktop\healthcare_fraud\.venv\Lib\site-packages\matplotlib\mpl-data\sample_data\msft.csv
c:\Users\suriy\Desktop\healthcare_fraud\.venv\Lib\site-packages\matplotlib\mpl-data\sample_data\Stocks.csv
c:\Users\suriy\Desktop\healthcare_fraud\.venv\Lib\site-packages\numpy\random\tests\data\mt19937-testset-1.csv
c:\Users\suriy\Desktop\he

In [4]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_TEST_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "test"
)

beneficiary_test = pd.read_csv(
    RAW_TEST_DIR
    / "Test_Beneficiarydata-1542969243754.csv"
)

inpatient_test = pd.read_csv(
    RAW_TEST_DIR
    / "Test_Inpatientdata-1542969243754.csv"
)

outpatient_test = pd.read_csv(
    RAW_TEST_DIR
    / "Test_Outpatientdata-1542969243754.csv"
)

print("=" * 70)
print("RAW TEST DATA LOADED")
print("=" * 70)

print("Beneficiary :", beneficiary_test.shape)
print("Inpatient   :", inpatient_test.shape)
print("Outpatient  :", outpatient_test.shape)

RAW TEST DATA LOADED
Beneficiary : (63968, 25)
Inpatient   : (9551, 30)
Outpatient  : (125841, 27)


In [5]:
import sys

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from prediction_processing import (
    preprocess_for_prediction
)

print("Production preprocessing imported successfully.")

Production preprocessing imported successfully.


In [6]:
X_production_test, provider_mapping, model_config = (
    preprocess_for_prediction(
        beneficiary_test,
        inpatient_test,
        outpatient_test
    )
)

FINAL 62-FEATURE PREDICTION PREPROCESSING
Saved feature count: 62
Model threshold: 0.4
Merged data shape: (135392, 56)
Final prediction shape: (135392, 62)
Expected feature count: 62
Missing values: 0


ValueError: Generated feature order does not match the saved model feature order.

In [7]:
print("Inpatient test :", inpatient_test.shape)
print("Outpatient test:", outpatient_test.shape)
print("Beneficiary test:", beneficiary_test.shape)

print(
    "\nExpected claim rows:",
    len(inpatient_test) + len(outpatient_test)
)

print(
    "Beneficiary duplicate BeneID:",
    beneficiary_test["BeneID"].duplicated().sum()
)

print(
    "Beneficiary rows:",
    len(beneficiary_test)
)

print(
    "Unique BeneID:",
    beneficiary_test["BeneID"].nunique()
)

Inpatient test : (9551, 30)
Outpatient test: (125841, 27)
Beneficiary test: (63968, 25)

Expected claim rows: 135392
Beneficiary duplicate BeneID: 0
Beneficiary rows: 63968
Unique BeneID: 63968


In [8]:
import pandas as pd

test_labels = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "raw"
    / "test"
    / "Test-1542969243754.csv"
)

print(test_labels.shape)
print(test_labels.columns.tolist())

(1353, 1)
['Provider']


In [9]:
test_providers = set(
    test_labels["Provider"]
    .dropna()
    .unique()
)

print(
    "Number of test providers:",
    len(test_providers)
)

Number of test providers: 1353


In [10]:
inpatient_test = inpatient_test[
    inpatient_test["Provider"].isin(test_providers)
].copy()

outpatient_test = outpatient_test[
    outpatient_test["Provider"].isin(test_providers)
].copy()

In [11]:
beneficiary_test_ids = set(
    pd.concat(
        [
            inpatient_test["BeneID"],
            outpatient_test["BeneID"]
        ]
    )
    .dropna()
)

beneficiary_test = beneficiary_test[
    beneficiary_test["BeneID"].isin(
        beneficiary_test_ids
    )
].copy()

In [12]:
print("=" * 70)
print("FILTERED TEST DATA")
print("=" * 70)

print("Inpatient test   :", inpatient_test.shape)
print("Outpatient test  :", outpatient_test.shape)
print("Beneficiary test :", beneficiary_test.shape)

print(
    "\nTotal claims:",
    len(inpatient_test) + len(outpatient_test)
)

FILTERED TEST DATA
Inpatient test   : (9551, 30)
Outpatient test  : (125841, 27)
Beneficiary test : (63968, 25)

Total claims: 135392


In [13]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_TRAIN_DIR = PROJECT_ROOT / "data" / "raw"

beneficiary_df = pd.read_csv(
    RAW_TRAIN_DIR / "Train_Beneficiarydata-1542865627584.csv"
)

inpatient_df = pd.read_csv(
    RAW_TRAIN_DIR / "Train_Inpatientdata-1542865627584.csv"
)

outpatient_df = pd.read_csv(
    RAW_TRAIN_DIR / "Train_Outpatientdata-1542865627584.csv"
)

provider_labels = pd.read_csv(
    RAW_TRAIN_DIR / "Train-1542865627584.csv"
)

print("Beneficiary:", beneficiary_df.shape)
print("Inpatient:", inpatient_df.shape)
print("Outpatient:", outpatient_df.shape)
print("Provider labels:", provider_labels.shape)

Beneficiary: (138556, 25)
Inpatient: (40474, 30)
Outpatient: (517737, 27)
Provider labels: (5410, 2)


In [14]:
from sklearn.model_selection import train_test_split

train_providers, test_providers = train_test_split(
    provider_labels["Provider"],
    test_size=0.20,
    random_state=42,
    stratify=provider_labels["PotentialFraud"]
)

print("=" * 70)
print("REPRODUCED ORIGINAL PROVIDER SPLIT")
print("=" * 70)

print("Train providers:", len(train_providers))
print("Test providers :", len(test_providers))

REPRODUCED ORIGINAL PROVIDER SPLIT
Train providers: 4328
Test providers : 1082


In [15]:
inpatient_test = inpatient_df[
    inpatient_df["Provider"].isin(test_providers)
].copy()

outpatient_test = outpatient_df[
    outpatient_df["Provider"].isin(test_providers)
].copy()

In [16]:
beneficiary_test_ids = set(
    pd.concat(
        [
            inpatient_test["BeneID"],
            outpatient_test["BeneID"]
        ]
    ).dropna()
)

beneficiary_test = beneficiary_df[
    beneficiary_df["BeneID"].isin(
        beneficiary_test_ids
    )
].copy()

In [17]:
print("=" * 70)
print("EXACT ORIGINAL TEST DATA")
print("=" * 70)

print("Inpatient test   :", inpatient_test.shape)
print("Outpatient test  :", outpatient_test.shape)
print("Beneficiary test :", beneficiary_test.shape)

print(
    "\nTotal test claims:",
    len(inpatient_test) + len(outpatient_test)
)

EXACT ORIGINAL TEST DATA
Inpatient test   : (8978, 30)
Outpatient test  : (124798, 27)
Beneficiary test : (61836, 25)

Total test claims: 133776


In [18]:
# ============================================================
# RUN NEW PRODUCTION 62-FEATURE PREPROCESSING
# ============================================================

X_production_test, provider_mapping, model_config = (
    preprocess_for_prediction(
        beneficiary_test,
        inpatient_test,
        outpatient_test
    )
)

FINAL 62-FEATURE PREDICTION PREPROCESSING
Saved feature count: 62
Model threshold: 0.4
Merged data shape: (133776, 56)
Final prediction shape: (133776, 62)
Expected feature count: 62
Missing values: 0


ValueError: Generated feature order does not match the saved model feature order.

In [20]:
X_production_test, provider_mapping, model_config = (
    preprocess_for_prediction(
        beneficiary_test,
        inpatient_test,
        outpatient_test
    )
)

FINAL 62-FEATURE PREDICTION PREPROCESSING
Saved feature count: 62
Model threshold: 0.4
Merged data shape: (133776, 56)
Final prediction shape: (133776, 62)
Expected feature count: 62
Missing values: 0


ValueError: Generated feature order does not match the saved model feature order.

In [1]:
import pandas as pd
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

In [3]:
from prediction_processing import preprocess_for_prediction

print("Production preprocessing imported successfully.")

Production preprocessing imported successfully.


In [4]:
import inspect
import prediction_processing

print(
    inspect.getsource(
        prediction_processing.preprocess_for_prediction
    )[-3000:]
)

in data.columns:

        raise ValueError(
            "Provider column is missing."
        )


    provider_mapping = data[
        ["Provider"]
    ].copy()


    provider_mapping.reset_index(
        drop=True,
        inplace=True
    )


    # ========================================================
    # BASIC FEATURES
    # ========================================================

    data = create_basic_features(
        data
    )


    # ========================================================
    # CREATE EXACT 62 FEATURES
    # ========================================================

    X_prediction = create_selected_features(
        data
    )


    # ========================================================
    # RESET INDEX
    # ========================================================

    X_prediction.reset_index(
        drop=True,
        inplace=True
    )


    provider_mapping.reset_index(
        drop=True,
        inplace=True
    )


    # =================

In [5]:
X_production_test, provider_mapping, model_config = (
    preprocess_for_prediction(
        beneficiary_test,
        inpatient_test,
        outpatient_test
    )
)

NameError: name 'beneficiary_test' is not defined

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
import sys

PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"

beneficiary_df = pd.read_csv(
    RAW_DIR / "Train_Beneficiarydata-1542865627584.csv"
)

inpatient_df = pd.read_csv(
    RAW_DIR / "Train_Inpatientdata-1542865627584.csv"
)

outpatient_df = pd.read_csv(
    RAW_DIR / "Train_Outpatientdata-1542865627584.csv"
)

provider_labels = pd.read_csv(
    RAW_DIR / "Train-1542865627584.csv"
)

print("Beneficiary:", beneficiary_df.shape)
print("Inpatient:", inpatient_df.shape)
print("Outpatient:", outpatient_df.shape)
print("Provider labels:", provider_labels.shape)

Beneficiary: (138556, 25)
Inpatient: (40474, 30)
Outpatient: (517737, 27)
Provider labels: (5410, 2)


In [2]:
train_providers, test_providers = train_test_split(
    provider_labels["Provider"],
    test_size=0.20,
    random_state=42,
    stratify=provider_labels["PotentialFraud"]
)

print("Train providers:", len(train_providers))
print("Test providers :", len(test_providers))

Train providers: 4328
Test providers : 1082


In [3]:
inpatient_test = inpatient_df[
    inpatient_df["Provider"].isin(test_providers)
].copy()

outpatient_test = outpatient_df[
    outpatient_df["Provider"].isin(test_providers)
].copy()

beneficiary_test_ids = set(
    pd.concat(
        [
            inpatient_test["BeneID"],
            outpatient_test["BeneID"]
        ]
    ).dropna()
)

beneficiary_test = beneficiary_df[
    beneficiary_df["BeneID"].isin(
        beneficiary_test_ids
    )
].copy()

print("=" * 70)
print("EXACT TEST DATA")
print("=" * 70)

print("Inpatient   :", inpatient_test.shape)
print("Outpatient  :", outpatient_test.shape)
print("Beneficiary :", beneficiary_test.shape)

print(
    "Total claims:",
    len(inpatient_test) + len(outpatient_test)
)

EXACT TEST DATA
Inpatient   : (8978, 30)
Outpatient  : (124798, 27)
Beneficiary : (61836, 25)
Total claims: 133776


In [4]:
inpatient_test = inpatient_df[
    inpatient_df["Provider"].isin(test_providers)
].copy()

outpatient_test = outpatient_df[
    outpatient_df["Provider"].isin(test_providers)
].copy()

beneficiary_test_ids = set(
    pd.concat(
        [
            inpatient_test["BeneID"],
            outpatient_test["BeneID"]
        ]
    ).dropna()
)

beneficiary_test = beneficiary_df[
    beneficiary_df["BeneID"].isin(
        beneficiary_test_ids
    )
].copy()

print("=" * 70)
print("EXACT TEST DATA")
print("=" * 70)

print("Inpatient   :", inpatient_test.shape)
print("Outpatient  :", outpatient_test.shape)
print("Beneficiary :", beneficiary_test.shape)

print(
    "Total claims:",
    len(inpatient_test) + len(outpatient_test)
)

EXACT TEST DATA
Inpatient   : (8978, 30)
Outpatient  : (124798, 27)
Beneficiary : (61836, 25)
Total claims: 133776


In [5]:
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from prediction_processing import preprocess_for_prediction

print("Production preprocessing imported successfully.")

Production preprocessing imported successfully.


In [6]:
X_production_test, provider_mapping, model_config = (
    preprocess_for_prediction(
        beneficiary_test,
        inpatient_test,
        outpatient_test
    )
)

FINAL 62-FEATURE PREDICTION PREPROCESSING
Saved feature count: 62
Model threshold: 0.4
Merged data shape: (133776, 55)


ValueError: Production 59-column checkpoint has different columns. Expected=59, generated=65, missing=[], extra=['claim_strt_year', 'adm_year', 'Difference', 'birth_year', 'IsAlive', 'Age_Category']

In [7]:
# ============================================================
# FINAL PRODUCTION vs ORIGINAL 62-FEATURE COMPARISON
# ============================================================

import pandas as pd
import numpy as np

# Load the original experiment test data
X_test_62 = pd.read_csv(
    "data/X_test_62.csv"
)

original = X_test_62.reset_index(drop=True)
production = X_production_test.reset_index(drop=True)

print("=" * 70)
print("ORIGINAL vs PRODUCTION 62-FEATURE DATA")
print("=" * 70)

print("Original shape   :", original.shape)
print("Production shape :", production.shape)

print(
    "Same columns     :",
    list(original.columns) == list(production.columns)
)

print(
    "Same row count   :",
    len(original) == len(production)
)

# ------------------------------------------------------------
# VALUE DIFFERENCE
# ------------------------------------------------------------

difference = (
    production.astype(float)
    - original.astype(float)
).abs()

different_cells = (
    difference > 1e-6
).sum().sum()

total_cells = difference.size

print()
print("=" * 70)
print("FEATURE VALUE COMPARISON")
print("=" * 70)

print(
    "Maximum absolute difference:",
    difference.max().max()
)

print(
    "Mean absolute difference:",
    difference.mean().mean()
)

print(
    "Different cells:",
    different_cells
)

print(
    "Percentage of cells different:",
    (different_cells / total_cells) * 100
)

ORIGINAL vs PRODUCTION 62-FEATURE DATA
Original shape   : (133776, 62)
Production shape : (133776, 62)
Same columns     : True
Same row count   : True

FEATURE VALUE COMPARISON
Maximum absolute difference: 534.0
Mean absolute difference: 0.1361872020242976
Different cells: 564540
Percentage of cells different: 6.806515272521037


In [8]:
# ============================================================
# FIND FEATURES STILL DIFFERING
# ============================================================

feature_comparison = []

for feature in original.columns:

    original_values = pd.to_numeric(
        original[feature],
        errors="coerce"
    ).fillna(0)

    production_values = pd.to_numeric(
        production[feature],
        errors="coerce"
    ).fillna(0)

    diff = (
        production_values - original_values
    ).abs()

    feature_comparison.append({
        "feature": feature,
        "max_difference": diff.max(),
        "mean_difference": diff.mean(),
        "different_values": (
            diff > 1e-6
        ).sum()
    })

feature_comparison = pd.DataFrame(
    feature_comparison
).sort_values(
    "mean_difference",
    ascending=False
)

print("=" * 90)
print("TOP DIFFERING FEATURES")
print("=" * 90)

print(
    feature_comparison.head(15).to_string(
        index=False
    )
)

TOP DIFFERING FEATURES
                                            feature  max_difference  mean_difference  different_values
                     mean_Hospital_Days_perProvider    2.342857e+01     2.383761e+00             65369
                          mean_Claim_Days_perBeneID    3.550000e+01     1.697053e+00             54584
           mean_Hospital_Days_perClmDiagnosisCode_1    3.181818e+01     1.159768e+00             42982
       mean_DeductibleAmtPaid_perAttendingPhysician    5.340000e+02     1.012474e+00              3958
                 mean_DeductibleAmtPaid_perProvider    5.340000e+02     5.778139e-01             37663
                                           Gender_0    1.000000e+00     5.773607e-01             77237
                               mean_Age_perProvider    1.313484e+00     5.565503e-01            133776
           mean_Hospital_Days_perAttendingPhysician    2.729412e+01     2.170816e-01              7113
                              mean_Risk_perProvide

In [9]:
# ============================================================
# FIND EXACT FEATURES CAUSING THE DIFFERENCE
# ============================================================

import pandas as pd
import numpy as np

original = X_test_62.reset_index(drop=True)
production = X_production_test.reset_index(drop=True)

feature_comparison = []

for feature in original.columns:

    original_values = pd.to_numeric(
        original[feature],
        errors="coerce"
    ).fillna(0)

    production_values = pd.to_numeric(
        production[feature],
        errors="coerce"
    ).fillna(0)

    difference = (
        production_values - original_values
    ).abs()

    feature_comparison.append({
        "feature": feature,
        "max_difference": difference.max(),
        "mean_difference": difference.mean(),
        "different_values": (
            difference > 1e-6
        ).sum()
    })

feature_comparison = pd.DataFrame(
    feature_comparison
)

feature_comparison = feature_comparison.sort_values(
    "mean_difference",
    ascending=False
).reset_index(drop=True)

print("=" * 90)
print("EXACT FEATURES CAUSING PRODUCTION vs ORIGINAL DIFFERENCE")
print("=" * 90)

print(
    feature_comparison.to_string(index=False)
)

EXACT FEATURES CAUSING PRODUCTION vs ORIGINAL DIFFERENCE
                                            feature  max_difference  mean_difference  different_values
        count_ClaimID_perProviderAttendingPhysician    4.726000e+03     8.111482e+02            125811
                    count_ClaimID_perProviderBeneID    2.637000e+03     5.010109e+02            133708
            count_ClaimID_perProviderOtherPhysician    1.872000e+03     3.214687e+02            126632
        count_ClaimID_perProviderClmDiagnosisCode_4    1.119000e+03     2.551384e+02            133426
        count_ClaimID_perProviderOperatingPhysician    8.330000e+02     1.770439e+02            131377
        count_ClaimID_perProviderClmDiagnosisCode_5    6.890000e+02     1.650923e+02            133041
        count_ClaimID_perProviderClmDiagnosisCode_6    4.630000e+02     1.189387e+02            132420
        count_ClaimID_perProviderClmDiagnosisCode_7    3.500000e+02     8.911185e+01            131841
        count_Cl

In [10]:
# ============================================================
# TOP DIFFERING FEATURES
# ============================================================

print("=" * 90)
print("TOP 15 DIFFERING FEATURES")
print("=" * 90)

print(
    feature_comparison.head(15).to_string(index=False)
)

TOP 15 DIFFERING FEATURES
                                     feature  max_difference  mean_difference  different_values
 count_ClaimID_perProviderAttendingPhysician     4726.000000       811.148158            125811
             count_ClaimID_perProviderBeneID     2637.000000       501.010936            133708
     count_ClaimID_perProviderOtherPhysician     1872.000000       321.468686            126632
 count_ClaimID_perProviderClmDiagnosisCode_4     1119.000000       255.138396            133426
 count_ClaimID_perProviderOperatingPhysician      833.000000       177.043947            131377
 count_ClaimID_perProviderClmDiagnosisCode_5      689.000000       165.092311            133041
 count_ClaimID_perProviderClmDiagnosisCode_6      463.000000       118.938704            132420
 count_ClaimID_perProviderClmDiagnosisCode_7      350.000000        89.111851            131841
 count_ClaimID_perProviderClmDiagnosisCode_8      286.000000        68.753394            130927
 count_ClaimID

In [8]:
import pandas as pd
import numpy as np

# ============================================================
# LOAD ORIGINAL AND PRODUCTION 59-FEATURE DATA
# ============================================================

original = pd.read_pickle(
    "data/original_test_claims_59.pkl"
)

production = pd.read_pickle(
    "../data/production_test_claims_59.pkl"
)

print("=" * 70)
print("ORIGINAL vs PRODUCTION — 59 FEATURE DATA")
print("=" * 70)

print("Original shape   :", original.shape)
print("Production shape :", production.shape)

print(
    "\nSame columns:",
    list(original.columns) == list(production.columns)
)

print(
    "Same ClaimID set:",
    set(original["ClaimID"])
    == set(production["ClaimID"])
)

print(
    "Same Provider set:",
    set(original["Provider"])
    == set(production["Provider"])
)

print(
    "Same BeneID set:",
    set(original["BeneID"])
    == set(production["BeneID"])
)

ORIGINAL vs PRODUCTION — 59 FEATURE DATA
Original shape   : (133776, 59)
Production shape : (133776, 63)

Same columns: False
Same ClaimID set: True
Same Provider set: True
Same BeneID set: True


In [9]:
# ============================================================
# ALIGN BY CLAIM ID
# ============================================================

original = (
    original
    .sort_values("ClaimID")
    .reset_index(drop=True)
)

production = (
    production
    .sort_values("ClaimID")
    .reset_index(drop=True)
)

print(
    "\nSame ClaimID order:",
    original["ClaimID"].equals(
        production["ClaimID"]
    )
)


Same ClaimID order: True


In [10]:
# ============================================================
# FIND DIFFERENCES
# ============================================================

results = []

for column in original.columns:

    if column == "ClaimID":
        continue

    a = original[column]
    b = production[column]

    if pd.api.types.is_numeric_dtype(a):

        a = pd.to_numeric(
            a,
            errors="coerce"
        )

        b = pd.to_numeric(
            b,
            errors="coerce"
        )

        diff = (
            a - b
        ).abs().fillna(0)

        results.append({
            "feature": column,
            "different_rows": int(
                (diff > 1e-9).sum()
            ),
            "max_difference": float(
                diff.max()
            ),
            "mean_difference": float(
                diff.mean()
            )
        })

    else:

        different = (
            a.fillna("__NA__").astype(str)
            !=
            b.fillna("__NA__").astype(str)
        )

        results.append({
            "feature": column,
            "different_rows": int(
                different.sum()
            ),
            "max_difference": None,
            "mean_difference": None
        })


comparison = (
    pd.DataFrame(results)
    .sort_values(
        "different_rows",
        ascending=False
    )
)

display(comparison)

KeyError: 'IsHospitalized'

In [11]:
# ============================================================
# FIND EXACT COLUMN DIFFERENCE
# ============================================================

original = pd.read_pickle(
    "data/original_test_claims_59.pkl"
)

production = pd.read_pickle(
    "../data/production_test_claims_59.pkl"
)

original_columns = list(original.columns)
production_columns = list(production.columns)

print("=" * 70)
print("COLUMN DIFFERENCE")
print("=" * 70)

print("\nOriginal column count   :", len(original_columns))
print("Production column count :", len(production_columns))

print("\nColumns in ORIGINAL but NOT production:")

for col in original_columns:
    if col not in production_columns:
        print("  -", col)

print("\nColumns in PRODUCTION but NOT original:")

for col in production_columns:
    if col not in original_columns:
        print("  +", col)

COLUMN DIFFERENCE

Original column count   : 59
Production column count : 63

Columns in ORIGINAL but NOT production:
  - IsHospitalized

Columns in PRODUCTION but NOT original:
  + IsHospitalized_x
  + IsHospitalized_y
  + Difference
  + IsAlive
  + Age_Category


In [12]:
# ============================================================
# SHOW BOTH COLUMN LISTS
# ============================================================

print("\n" + "=" * 70)
print("ORIGINAL COLUMNS")
print("=" * 70)

for i, col in enumerate(original.columns, start=1):
    print(i, col)


print("\n" + "=" * 70)
print("PRODUCTION COLUMNS")
print("=" * 70)

for i, col in enumerate(production.columns, start=1):
    print(i, col)


ORIGINAL COLUMNS
1 BeneID
2 ClaimID
3 ClaimStartDt
4 ClaimEndDt
5 Provider
6 InscClaimAmtReimbursed
7 AttendingPhysician
8 OperatingPhysician
9 OtherPhysician
10 AdmissionDt
11 ClmAdmitDiagnosisCode
12 DeductibleAmtPaid
13 DischargeDt
14 DiagnosisGroupCode
15 ClmDiagnosisCode_1
16 ClmDiagnosisCode_2
17 ClmDiagnosisCode_3
18 ClmDiagnosisCode_4
19 ClmDiagnosisCode_5
20 ClmDiagnosisCode_6
21 ClmDiagnosisCode_7
22 ClmDiagnosisCode_8
23 ClmDiagnosisCode_9
24 ClmDiagnosisCode_10
25 ClmProcedureCode_1
26 ClmProcedureCode_2
27 ClmProcedureCode_3
28 ClmProcedureCode_4
29 ClmProcedureCode_5
30 ClmProcedureCode_6
31 DOB
32 DOD
33 Gender
34 Race
35 RenalDiseaseIndicator
36 State
37 County
38 NoOfMonths_PartACov
39 NoOfMonths_PartBCov
40 ChronicCond_Alzheimer
41 ChronicCond_Heartfailure
42 ChronicCond_KidneyDisease
43 ChronicCond_Cancer
44 ChronicCond_ObstrPulmonary
45 ChronicCond_Depression
46 ChronicCond_Diabetes
47 ChronicCond_IschemicHeart
48 ChronicCond_Osteoporasis
49 ChronicCond_rheumatoida

In [8]:
import pandas as pd
import numpy as np

# ============================================================
# LOAD ORIGINAL 62-FEATURE TEST DATA
# ============================================================

X_test_62 = pd.read_csv(
    "data/X_test_62.csv"
)

print("=" * 70)
print("ORIGINAL vs PRODUCTION 62-FEATURE DATA")
print("=" * 70)

print(
    "Production shape:",
    X_production_test.shape
)

print(
    "Saved test shape:",
    X_test_62.shape
)

print(
    "Same columns:",
    list(X_production_test.columns)
    == list(X_test_62.columns)
)

print(
    "Same row count:",
    len(X_production_test)
    == len(X_test_62)
)

# ============================================================
# NUMERICAL COMPARISON
# ============================================================

production_values = (
    X_production_test
    .reset_index(drop=True)
    .astype(float)
)

saved_values = (
    X_test_62
    .reset_index(drop=True)
    .astype(float)
)

difference = (
    production_values - saved_values
).abs()

different_cells = (
    difference > 1e-6
).sum().sum()

total_cells = difference.size

print("\nMaximum absolute difference:")
print(difference.max().max())

print("\nMean absolute difference:")
print(difference.mean().mean())

print("\nTotal cells different:")
print(different_cells)

print("\nPercentage of cells different:")
print(
    (different_cells / total_cells) * 100
)

ORIGINAL vs PRODUCTION 62-FEATURE DATA
Production shape: (133776, 62)
Saved test shape: (133776, 62)
Same columns: True
Same row count: True

Maximum absolute difference:
154633.63636363635

Mean absolute difference:
233.68873287863244

Total cells different:
5110566

Percentage of cells different:
61.61679514334989


In [9]:
# ============================================================
# COMPARE PRODUCTION FEATURES WITH ORIGINAL TEST FEATURES
# ============================================================

original_X_test = pd.read_csv(
    PROJECT_ROOT
    / "experiments"
    / "data"
    / "X_test_62.csv"
)

print("=" * 70)
print("PRODUCTION vs ORIGINAL 62-FEATURE DATA")
print("=" * 70)

print("Original shape   :", original_X_test.shape)
print("Production shape :", X_production_test.shape)

print(
    "\nSame columns:",
    list(original_X_test.columns)
    == list(X_production_test.columns)
)

# Make sure indexes do not affect comparison
original_X_test = original_X_test.reset_index(drop=True)
X_production_test = X_production_test.reset_index(drop=True)

# Numeric difference
difference = (
    original_X_test.astype(float)
    - X_production_test.astype(float)
).abs()

print(
    "\nMaximum absolute difference:",
    difference.to_numpy().max()
)

print(
    "Mean absolute difference:",
    difference.to_numpy().mean()
)

print(
    "Number of different values:",
    (difference > 1e-10).sum().sum()
)

PRODUCTION vs ORIGINAL 62-FEATURE DATA
Original shape   : (133776, 62)
Production shape : (133776, 62)

Same columns: True

Maximum absolute difference: 154633.63636363635
Mean absolute difference: 233.68873287863246
Number of different values: 5110566


In [10]:
# ============================================================
# FIND WHICH FEATURES HAVE DIFFERENT VALUES
# ============================================================

comparison = pd.DataFrame({
    "feature": X_production_test.columns,
    "max_difference": [
        (
            original_X_test[col].astype(float)
            -
            X_production_test[col].astype(float)
        ).abs().max()
        for col in X_production_test.columns
    ],
    "mean_difference": [
        (
            original_X_test[col].astype(float)
            -
            X_production_test[col].astype(float)
        ).abs().mean()
        for col in X_production_test.columns
    ],
    "different_values": [
        (
            (
                original_X_test[col].astype(float)
                -
                X_production_test[col].astype(float)
            ).abs()
            > 1e-10
        ).sum()
        for col in X_production_test.columns
    ]
})

comparison = comparison.sort_values(
    "max_difference",
    ascending=False
)

print("=" * 70)
print("FEATURE VALUE COMPARISON")
print("=" * 70)

print(comparison.to_string(index=False))

FEATURE VALUE COMPARISON
                                            feature  max_difference  mean_difference  different_values
mean_IPAnnualReimbursementAmt_perAttendingPhysician   154633.636364      4520.378635            132327
  mean_InscClaimAmtReimbursed_perAttendingPhysician   124690.476190      1372.554348            133560
          mean_IPAnnualReimbursementAmt_perProvider    90044.718460      2157.978766            132862
            mean_InscClaimAmtReimbursed_perProvider    56307.272727      1046.714620            132862
   mean_IPAnnualDeductibleAmt_perAttendingPhysician    37916.000000       450.978649            131814
mean_IPAnnualReimbursementAmt_perClmProcedureCode_5    13255.000000         0.396334                 4
          mean_OPAnnualReimbursementAmt_perProvider    12630.650000       498.443616            132862
             mean_IPAnnualDeductibleAmt_perProvider    11246.067387       226.947409            132858
  mean_InscClaimAmtReimbursed_perClmProcedureCod

In [11]:
# ============================================================
# CHECK RAW CLAIM ROW ORDER
# ============================================================

original_claims = pd.concat(
    [
        inpatient_test,
        outpatient_test
    ],
    ignore_index=True
)

print("=" * 70)
print("RAW CLAIM ORDER CHECK")
print("=" * 70)

print(
    "Original claim rows:",
    len(original_claims)
)

print(
    "Production merged rows:",
    len(X_production_test)
)

print("\nFirst 10 ClaimIDs:")
print(
    original_claims["ClaimID"]
    .head(10)
    .tolist()
)

RAW CLAIM ORDER CHECK
Original claim rows: 133776
Production merged rows: 133776

First 10 ClaimIDs:
['CLM63689', 'CLM31519', 'CLM65412', 'CLM57153', 'CLM38115', 'CLM41414', 'CLM60246', 'CLM48802', 'CLM60118', 'CLM34789']


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from prediction_processing import preprocess_for_prediction

print("Production preprocessing imported successfully.")

Production preprocessing imported successfully.


In [9]:
X_production_test, provider_mapping, model_config = (
    preprocess_for_prediction(
        beneficiary_test,
        inpatient_test,
        outpatient_test
    )
)

FINAL 62-FEATURE PREDICTION PREPROCESSING
Saved feature count: 62
Model threshold: 0.4
Merged data shape: (133776, 56)
Final prediction shape: (133776, 62)
Expected feature count: 62
Missing values: 0

62-FEATURE PREPROCESSING PASSED


In [11]:
# ============================================================
# LOAD ORIGINAL 62-FEATURE TEST DATA
# ============================================================

original_test = pd.read_csv(
    "data/X_test_62.csv"
)

original_test = original_test[
    X_production_test.columns
].copy()

production_test = X_production_test[
    original_test.columns
].copy()

# ============================================================
# COMPARE
# ============================================================

difference = (
    production_test.astype(float)
    - original_test.astype(float)
).abs()

print("=" * 70)
print("PRODUCTION vs ORIGINAL 62-FEATURE DATA")
print("=" * 70)

print("Original shape   :", original_test.shape)
print("Production shape :", production_test.shape)

print(
    "Same columns     :",
    list(original_test.columns)
    == list(production_test.columns)
)

print(
    "Maximum absolute difference :",
    difference.max().max()
)

print(
    "Mean absolute difference    :",
    difference.mean().mean()
)

print(
    "Number of different values :",
    (difference > 1e-9).sum().sum()
)

PRODUCTION vs ORIGINAL 62-FEATURE DATA
Original shape   : (133776, 62)
Production shape : (133776, 62)
Same columns     : True
Maximum absolute difference : 154633.63636363635
Mean absolute difference    : 233.68873287863244
Number of different values : 5110566


In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from prediction_processing import preprocess_for_prediction

print("Production preprocessing imported successfully.")

Production preprocessing imported successfully.


In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

TEST_DIR = PROJECT_ROOT / "data" / "raw" / "test"

beneficiary_test = pd.read_csv(
    TEST_DIR / "Test_Beneficiarydata-1542969243754.csv"
)

inpatient_test = pd.read_csv(
    TEST_DIR / "Test_Inpatientdata-1542969243754.csv"
)

outpatient_test = pd.read_csv(
    TEST_DIR / "Test_Outpatientdata-1542969243754.csv"
)

test_labels = pd.read_csv(
    TEST_DIR / "Test-1542969243754.csv"
)

print("Beneficiary :", beneficiary_test.shape)
print("Inpatient   :", inpatient_test.shape)
print("Outpatient  :", outpatient_test.shape)
print("Test labels :", test_labels.shape)

Beneficiary : (63968, 25)
Inpatient   : (9551, 30)
Outpatient  : (125841, 27)
Test labels : (1353, 1)


In [4]:
X_production_test, provider_mapping, model_config = (
    preprocess_for_prediction(
        beneficiary_test,
        inpatient_test,
        outpatient_test
    )
)

print("Production features:", X_production_test.shape)

FINAL 62-FEATURE PREDICTION PREPROCESSING
Saved feature count: 62
Model threshold: 0.4
Merged data shape: (135392, 55)
Basic feature checkpoint: (135392, 59) (expected 59 columns)
59-column names match original experiment: True
59-column order restored to original experiment: True


c:\Users\suriy\Desktop\healthcare_fraud\src\prediction_processing.py:484: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[feature_name] = (
c:\Users\suriy\Desktop\healthcare_fraud\src\prediction_processing.py:484: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[feature_name] = (
c:\Users\suriy\Desktop\healthcare_fraud\src\prediction_processing.py:484: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all column

Final prediction shape: (135392, 62)
Expected feature count: 62
Missing values: 0

62-FEATURE PREPROCESSING PASSED
Production features: (135392, 62)


In [7]:
# ============================================================
# LOAD SAVED FINAL CATBOOST MODEL
# ============================================================

from pathlib import Path
from catboost import CatBoostClassifier
import pickle

PROJECT_ROOT = Path.cwd().parent

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "v2_62_features"
)

MODEL_PATH = MODEL_DIR / "catboost_model.cbm"
FEATURE_PATH = MODEL_DIR / "feature_columns.pkl"
CONFIG_PATH = MODEL_DIR / "model_config.pkl"

print("=" * 70)
print("LOADING SAVED CATBOOST MODEL")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load CatBoost model
# ------------------------------------------------------------

model = CatBoostClassifier()

model.load_model(
    str(MODEL_PATH)
)

print("Model loaded successfully.")

# ------------------------------------------------------------
# 2. Load saved feature columns
# ------------------------------------------------------------

with open(FEATURE_PATH, "rb") as f:
    saved_feature_columns = pickle.load(f)

print(
    "Saved feature count:",
    len(saved_feature_columns)
)

# ------------------------------------------------------------
# 3. Load model configuration
# ------------------------------------------------------------

with open(CONFIG_PATH, "rb") as f:
    model_config = pickle.load(f)

print("Model configuration:")
print(model_config)

# ------------------------------------------------------------
# 4. Verify model / feature count
# ------------------------------------------------------------

if len(saved_feature_columns) != 62:
    raise ValueError(
        f"Expected 62 saved features, "
        f"found {len(saved_feature_columns)}"
    )

if X_production_test.shape[1] != 62:
    raise ValueError(
        f"Production data has "
        f"{X_production_test.shape[1]} features, "
        f"expected 62"
    )

if list(X_production_test.columns) != list(
    saved_feature_columns
):
    raise ValueError(
        "Production feature order does not match "
        "the saved CatBoost feature order."
    )

print("\nFeature order: MATCH")
print("Feature count : 62")
print("Model loaded  : YES")

print("\n" + "=" * 70)
print("MODEL READY FOR PREDICTION")
print("=" * 70)

LOADING SAVED CATBOOST MODEL
Model loaded successfully.
Saved feature count: 62
Model configuration:
{'model_name': 'CatBoost', 'feature_count': 62, 'threshold': 0.4, 'random_seed': 42, 'depth': 5, 'iterations': 200, 'learning_rate': 0.1, 'test_accuracy': 0.9127, 'test_precision': 0.915, 'test_recall': 0.8868, 'test_f1': 0.9007, 'test_roc_auc': 0.9542, 'test_pr_auc': 0.9584}

Feature order: MATCH
Feature count : 62
Model loaded  : YES

MODEL READY FOR PREDICTION


In [9]:
# ============================================================
# DIAGNOSTIC: ORIGINAL 62-FEATURE TEST vs PRODUCTION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 70)
print("DIAGNOSTIC: ORIGINAL TEST vs PRODUCTION")
print("=" * 70)

# ------------------------------------------------------------
# Find project root safely
# ------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "experiments").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "experiments").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path(
        r"C:\Users\suriy\Desktop\healthcare_fraud"
    )

ORIGINAL_PATH = (
    PROJECT_ROOT
    / "experiments"
    / "data"
    / "X_test_62.csv"
)

print("Project root :", PROJECT_ROOT)
print("Original file:", ORIGINAL_PATH)

if not ORIGINAL_PATH.exists():
    raise FileNotFoundError(
        f"Original 62-feature file not found:\n"
        f"{ORIGINAL_PATH}"
    )

# ------------------------------------------------------------
# Load ORIGINAL test features
# ------------------------------------------------------------

X_original = pd.read_csv(
    ORIGINAL_PATH
)

# ------------------------------------------------------------
# Match saved feature order
# ------------------------------------------------------------

X_original = X_original[
    saved_feature_columns
].copy()

X_production = X_production_test[
    saved_feature_columns
].copy()

print()
print("Original shape   :", X_original.shape)
print("Production shape :", X_production.shape)

# ------------------------------------------------------------
# Basic statistics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE DISTRIBUTION COMPARISON")
print("=" * 70)

comparison = []

for feature in saved_feature_columns:

    original_values = pd.to_numeric(
        X_original[feature],
        errors="coerce"
    ).fillna(0)

    production_values = pd.to_numeric(
        X_production[feature],
        errors="coerce"
    ).fillna(0)

    comparison.append({
        "feature": feature,

        "original_mean": original_values.mean(),
        "production_mean": production_values.mean(),

        "original_std": original_values.std(),
        "production_std": production_values.std(),

        "original_min": original_values.min(),
        "production_min": production_values.min(),

        "original_max": original_values.max(),
        "production_max": production_values.max(),

        "mean_difference": abs(
            original_values.mean()
            - production_values.mean()
        )
    })

comparison_df = pd.DataFrame(comparison)

comparison_df = comparison_df.sort_values(
    "mean_difference",
    ascending=False
)

display(
    comparison_df.head(20)
)

DIAGNOSTIC: ORIGINAL TEST vs PRODUCTION
Project root : c:\Users\suriy\Desktop\healthcare_fraud
Original file: c:\Users\suriy\Desktop\healthcare_fraud\experiments\data\X_test_62.csv

Original shape   : (133776, 62)
Production shape : (135392, 62)

FEATURE DISTRIBUTION COMPARISON


,feature,original_mean,production_mean,original_std,production_std,original_min,production_min,original_max,production_max,mean_difference
59,mean_IPAnnualReimbursementAmt_perClmProcedureC...,0.198167,5271.108559,51.251196,209.724075,0.0,5270.303937,13255.0,63000.000000,5270.910392
58,mean_InscClaimAmtReimbursed_perClmProcedureCode_5,0.089702,981.307906,23.199334,213.392988,0.0,980.487776,6000.0,57000.000000,981.218204
16,count_ClaimID_perProviderAttendingPhysician,944.097753,98.020755,1076.415229,170.924420,1.0,1.000000,4726.0,939.000000,846.076998
6,count_ClaimID_perProviderClmProcedureCode_5,0.015623,737.810210,0.124013,871.218966,0.0,1.000000,1.0,3250.000000,737.794587
1,count_ClaimID_perProviderClmProcedureCode_4,0.136026,737.620628,0.368968,871.079541,0.0,1.000000,2.0,3249.000000,737.484602
2,count_ClaimID_perProviderClmProcedureCode_3,1.064780,736.538496,2.059084,870.690069,0.0,1.000000,8.0,3247.000000,735.473716
14,count_ClaimID_perProviderClmProcedureCode_2,5.226154,730.382401,9.328490,867.913842,0.0,1.000000,39.0,3229.000000,725.156247
12,count_ClaimID_perProviderClmDiagnosisCode_10,5.726244,729.942655,8.308521,867.113198,0.0,1.000000,33.0,3229.000000,724.216412
17,count_ClaimID_perProviderClmProcedureCode_1,21.623983,707.864793,37.977732,856.178809,0.0,1.000000,155.0,3142.000000,686.240809
18,count_ClaimID_perProviderDiagnosisGroupCode,36.770676,689.926672,64.489452,846.891587,0.0,1.000000,248.0,3065.000000,653.155996


In [10]:
print("=" * 70)
print("CURRENT RAW TEST DATA")
print("=" * 70)

print("Inpatient :", inpatient_test.shape)
print("Outpatient:", outpatient_test.shape)
print("Beneficiary:", beneficiary_test.shape)

print()

claims_now = pd.concat(
    [
        inpatient_test[["ClaimID"]],
        outpatient_test[["ClaimID"]]
    ],
    ignore_index=True
)

print("Total claims:", len(claims_now))

print(
    "Unique ClaimID:",
    claims_now["ClaimID"].nunique()
)

print(
    "Duplicate ClaimID:",
    claims_now["ClaimID"].duplicated().sum()
)

CURRENT RAW TEST DATA
Inpatient : (9551, 30)
Outpatient: (125841, 27)
Beneficiary: (63968, 25)

Total claims: 135392
Unique ClaimID: 135392
Duplicate ClaimID: 0


In [11]:
# ============================================================
# CHECK MODEL ON ITS ORIGINAL 62-FEATURE TEST DATA
# ============================================================

print("=" * 70)
print("MODEL SANITY CHECK")
print("=" * 70)

# X_original = experiments/data/X_test_62.csv

original_probability = model.predict_proba(
    X_original
)[:, 1]

original_prediction = (
    original_probability >= threshold
).astype(int)

print("\nOriginal development-test rows:")
print(len(X_original))

print("\nThreshold:")
print(threshold)

print("\nPredicted classes:")
print(
    pd.Series(
        original_prediction,
        name="Fraud_Prediction"
    ).value_counts().sort_index()
)

print("\nPredicted fraud percentage:")
print(
    original_prediction.mean() * 100
)

print("\nProbability statistics:")
print(
    pd.Series(original_probability).describe()
)

print("=" * 70)

MODEL SANITY CHECK

Original development-test rows:
133776

Threshold:
0.4

Predicted classes:
Fraud_Prediction
0    75916
1    57860
Name: count, dtype: int64

Predicted fraud percentage:
43.251405334290155

Probability statistics:
count    133776.000000
mean          0.437178
std           0.454709
min           0.000005
25%           0.009872
50%           0.147742
75%           0.997581
max           0.999977
dtype: float64


In [12]:
# ============================================================
# VERIFY ORIGINAL MODEL PERFORMANCE
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("=" * 70)
print("ORIGINAL TEST SET MODEL PERFORMANCE")
print("=" * 70)

# ------------------------------------------------------------
# LOAD ORIGINAL TEST LABELS
# ------------------------------------------------------------

Y_TEST_PATH = (
    PROJECT_ROOT
    / "experiments"
    / "data"
    / "y_test_62.csv"
)

print("Label file:")
print(Y_TEST_PATH)

y_test_original = pd.read_csv(
    Y_TEST_PATH
)["PotentialFraud"]

# ------------------------------------------------------------
# SAFETY CHECKS
# ------------------------------------------------------------

print("\nX_test rows :", len(X_original))
print("y_test rows :", len(y_test_original))

if len(X_original) != len(y_test_original):
    raise ValueError(
        "X_test and y_test have different row counts."
    )

# ------------------------------------------------------------
# TRUE LABEL DISTRIBUTION
# ------------------------------------------------------------

print("\nActual class distribution:")
print(
    y_test_original.value_counts()
)

print("\nActual fraud percentage:")

# Handle either 0/1 or string labels
if y_test_original.dtype == object:
    print(
        y_test_original.value_counts(normalize=True)
    )
else:
    print(
        y_test_original.mean() * 100
    )

# ------------------------------------------------------------
# MODEL METRICS
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test_original,
    original_prediction
)

precision = precision_score(
    y_test_original,
    original_prediction,
    zero_division=0
)

recall = recall_score(
    y_test_original,
    original_prediction,
    zero_division=0
)

f1 = f1_score(
    y_test_original,
    original_prediction,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test_original,
    original_probability
)

pr_auc = average_precision_score(
    y_test_original,
    original_probability
)

cm = confusion_matrix(
    y_test_original,
    original_prediction
)

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL ORIGINAL TEST METRICS")
print("=" * 70)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_test_original,
        original_prediction,
        zero_division=0
    )
)

print("=" * 70)

ORIGINAL TEST SET MODEL PERFORMANCE
Label file:
c:\Users\suriy\Desktop\healthcare_fraud\experiments\data\y_test_62.csv

X_test rows : 133776
y_test rows : 133776

Actual class distribution:
PotentialFraud
0    74076
1    59700
Name: count, dtype: int64

Actual fraud percentage:
44.62683889486904

FINAL ORIGINAL TEST METRICS
Accuracy : 0.9127
Precision: 0.9150
Recall   : 0.8868
F1       : 0.9007
ROC-AUC  : 0.9542
PR-AUC   : 0.9584

Confusion Matrix:
[[69158  4918]
 [ 6758 52942]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.93      0.92     74076
           1       0.92      0.89      0.90     59700

    accuracy                           0.91    133776
   macro avg       0.91      0.91      0.91    133776
weighted avg       0.91      0.91      0.91    133776



In [1]:
# ============================================================
# LOAD DEVELOPMENT DATA FOR PROVIDER-LEVEL EVALUATION
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    r"C:\Users\suriy\Desktop\healthcare_fraud"
)

TRAIN_DIR = PROJECT_ROOT / "data" / "raw"

beneficiary_dev = pd.read_csv(
    TRAIN_DIR / "Train_Beneficiarydata-1542865627584.csv"
)

inpatient_dev = pd.read_csv(
    TRAIN_DIR / "Train_Inpatientdata-1542865627584.csv"
)

outpatient_dev = pd.read_csv(
    TRAIN_DIR / "Train_Outpatientdata-1542865627584.csv"
)

provider_labels_dev = pd.read_csv(
    TRAIN_DIR / "Train-1542865627584.csv"
)

print("=" * 70)
print("DEVELOPMENT DATA LOADED")
print("=" * 70)

print("Beneficiary :", beneficiary_dev.shape)
print("Inpatient   :", inpatient_dev.shape)
print("Outpatient  :", outpatient_dev.shape)
print("Labels      :", provider_labels_dev.shape)

DEVELOPMENT DATA LOADED
Beneficiary : (138556, 25)
Inpatient   : (40474, 30)
Outpatient  : (517737, 27)
Labels      : (5410, 2)


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from prediction_processing import preprocess_for_prediction

print("Production preprocessing imported successfully.")

Production preprocessing imported successfully.


In [5]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print("PROJECT ROOT:")
print(PROJECT_ROOT)

print("\nCSV FILES FOUND:")
for file in PROJECT_ROOT.rglob("*.csv"):
    print(file)

PROJECT ROOT:
c:\Users\suriy\Desktop\healthcare_fraud

CSV FILES FOUND:
c:\Users\suriy\Desktop\healthcare_fraud\X_test.csv
c:\Users\suriy\Desktop\healthcare_fraud\X_train.csv
c:\Users\suriy\Desktop\healthcare_fraud\X_val.csv
c:\Users\suriy\Desktop\healthcare_fraud\y_test.csv
c:\Users\suriy\Desktop\healthcare_fraud\y_train.csv
c:\Users\suriy\Desktop\healthcare_fraud\y_val.csv
c:\Users\suriy\Desktop\healthcare_fraud\outputs\claim_predictions.csv
c:\Users\suriy\Desktop\healthcare_fraud\outputs\provider_predictions.csv
c:\Users\suriy\Desktop\healthcare_fraud\outputs\unseen_test_provider_predictions.csv
c:\Users\suriy\Desktop\healthcare_fraud\.venv\Lib\site-packages\matplotlib\mpl-data\sample_data\data_x_x2_x3.csv
c:\Users\suriy\Desktop\healthcare_fraud\.venv\Lib\site-packages\matplotlib\mpl-data\sample_data\msft.csv
c:\Users\suriy\Desktop\healthcare_fraud\.venv\Lib\site-packages\matplotlib\mpl-data\sample_data\Stocks.csv
c:\Users\suriy\Desktop\healthcare_fraud\.venv\Lib\site-packages\numpy

In [6]:
# ============================================================
# LOAD OFFICIAL UNSEEN TEST DATA
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

TEST_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "test"
)

beneficiary_test = pd.read_csv(
    TEST_DIR / "Test_Beneficiarydata-1542969243754.csv",
    low_memory=False
)

inpatient_test = pd.read_csv(
    TEST_DIR / "Test_Inpatientdata-1542969243754.csv",
    low_memory=False
)

outpatient_test = pd.read_csv(
    TEST_DIR / "Test_Outpatientdata-1542969243754.csv",
    low_memory=False
)

print("=" * 70)
print("OFFICIAL UNSEEN TEST DATA LOADED")
print("=" * 70)

print("Beneficiary :", beneficiary_test.shape)
print("Inpatient   :", inpatient_test.shape)
print("Outpatient  :", outpatient_test.shape)

print(
    "Total claims:",
    len(inpatient_test) + len(outpatient_test)
)

print(
    "Unique ClaimID:",
    pd.concat(
        [
            inpatient_test[["ClaimID"]],
            outpatient_test[["ClaimID"]]
        ],
        ignore_index=True
    )["ClaimID"].nunique()
)

print("=" * 70)

OFFICIAL UNSEEN TEST DATA LOADED
Beneficiary : (63968, 25)
Inpatient   : (9551, 30)
Outpatient  : (125841, 27)
Total claims: 135392
Unique ClaimID: 135392


In [7]:
# ============================================================
# RUN CORRECTED PRODUCTION PREPROCESSING
# ============================================================

X_production_test, provider_mapping, model_config = (
    preprocess_for_prediction(
        beneficiary_test,
        inpatient_test,
        outpatient_test
    )
)

FINAL 62-FEATURE PREDICTION PREPROCESSING
Saved feature count: 62
Model threshold: 0.4
Merged data shape: (135392, 54)
Final prediction shape: (135392, 62)
Expected feature count: 62
Missing values: 0
62-FEATURE PREPROCESSING PASSED


In [8]:
# ============================================================
# FINAL TRAINING-vs-PRODUCTION FEATURE DISTRIBUTION CHECK
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# LOAD THE ORIGINAL DEVELOPMENT TEST FEATURES
# ------------------------------------------------------------

original_test = pd.read_csv(
    PROJECT_ROOT
    / "experiments"
    / "data"
    / "X_test_62.csv"
)

print("=" * 70)
print("DEVELOPMENT TEST vs OFFICIAL UNSEEN TEST")
print("=" * 70)

print(
    "Development X_test shape:",
    original_test.shape
)

print(
    "Official production shape:",
    X_production_test.shape
)

print(
    "Same columns:",
    list(original_test.columns)
    == list(X_production_test.columns)
)

# ------------------------------------------------------------
# COMPARE FEATURE DISTRIBUTIONS
# ------------------------------------------------------------

comparison = []

for feature in original_test.columns:

    original_values = pd.to_numeric(
        original_test[feature],
        errors="coerce"
    )

    production_values = pd.to_numeric(
        X_production_test[feature],
        errors="coerce"
    )

    comparison.append({

        "feature": feature,

        "development_mean":
            original_values.mean(),

        "production_mean":
            production_values.mean(),

        "development_std":
            original_values.std(),

        "production_std":
            production_values.std(),

        "development_min":
            original_values.min(),

        "production_min":
            production_values.min(),

        "development_max":
            original_values.max(),

        "production_max":
            production_values.max(),

        "mean_difference":
            abs(
                original_values.mean()
                - production_values.mean()
            )
    })


distribution_comparison = pd.DataFrame(
    comparison
)

distribution_comparison = (
    distribution_comparison
    .sort_values(
        "mean_difference",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# SHOW TOP DIFFERENCES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP 20 FEATURE DISTRIBUTION DIFFERENCES")
print("=" * 70)

display(
    distribution_comparison.head(20)
)


# ------------------------------------------------------------
# SPECIFIC PROVIDER FEATURES
# ------------------------------------------------------------

provider_features = [
    feature
    for feature in original_test.columns
    if (
        "perProvider" in feature
        or "perAttendingPhysician" in feature
        or "perBeneID" in feature
    )
]

print("\n" + "=" * 70)
print("PROVIDER-RELATED FEATURE DISTRIBUTIONS")
print("=" * 70)

display(
    distribution_comparison[
        distribution_comparison["feature"].isin(
            provider_features
        )
    ]
)


# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DISTRIBUTION CHECK COMPLETED")
print("=" * 70)

print(
    "Features checked:",
    len(distribution_comparison)
)

print(
    "Features with identical mean:",
    (
        distribution_comparison["mean_difference"]
        < 1e-10
    ).sum()
)

print(
    "Features with mean difference > 1:",
    (
        distribution_comparison["mean_difference"]
        > 1
    ).sum()
)

print("=" * 70)

DEVELOPMENT TEST vs OFFICIAL UNSEEN TEST
Development X_test shape: (133776, 62)
Official production shape: (135392, 62)
Same columns: True

TOP 20 FEATURE DISTRIBUTION DIFFERENCES


,feature,development_mean,production_mean,development_std,production_std,development_min,production_min,development_max,production_max,mean_difference
0,count_ClaimID_perProvider,946.552715,737.812382,1079.168407,871.217319,1.0,1.0,4739.0,3250.000000,208.740333
1,count_ClaimID_perProviderAttendingPhysician,944.097753,735.963351,1076.415229,869.004843,1.0,1.0,4726.0,3241.000000,208.134402
2,count_ClaimID_perProviderBeneID,503.865350,385.057773,566.571123,462.549041,1.0,1.0,2638.0,2552.000000,118.807577
3,mean_IPAnnualReimbursementAmt_perAttendingPhys...,5142.679105,5253.360317,5452.612573,5699.423338,-640.0,-400.0,155600.0,144000.000000,110.681211
4,mean_IPAnnualReimbursementAmt_perProvider,5160.864729,5271.108559,2357.120652,2669.241053,0.0,0.0,96000.0,65800.000000,110.243830
5,count_ClaimID_perProviderOtherPhysician,356.160133,265.884713,422.440361,326.364341,0.0,0.0,1872.0,1234.000000,90.275421
6,count_ClaimID_perProviderClmDiagnosisCode_4,256.087557,198.330248,279.705673,233.094922,0.0,0.0,1119.0,904.000000,57.757308
7,count_ClaimID_perProviderOperatingPhysician,182.315101,138.226010,201.834851,162.340525,0.0,0.0,833.0,638.000000,44.089091
8,mean_InscClaimAmtReimbursed_perAttendingPhysician,936.272500,978.750369,2656.029972,2779.921369,0.0,0.0,125000.0,70000.000000,42.477869
9,mean_InscClaimAmtReimbursed_perProvider,939.383372,981.307906,1427.134188,1654.221667,0.0,0.0,57000.0,57000.000000,41.924534



PROVIDER-RELATED FEATURE DISTRIBUTIONS


,feature,development_mean,production_mean,development_std,production_std,development_min,production_min,development_max,production_max,mean_difference
0,count_ClaimID_perProvider,946.552715,737.812382,1079.168407,871.217319,1.000000,1.000000,4739.000000,3250.000000,208.740333
1,count_ClaimID_perProviderAttendingPhysician,944.097753,735.963351,1076.415229,869.004843,1.000000,1.000000,4726.000000,3241.000000,208.134402
2,count_ClaimID_perProviderBeneID,503.865350,385.057773,566.571123,462.549041,1.000000,1.000000,2638.000000,2552.000000,118.807577
3,mean_IPAnnualReimbursementAmt_perAttendingPhys...,5142.679105,5253.360317,5452.612573,5699.423338,-640.000000,-400.000000,155600.000000,144000.000000,110.681211
4,mean_IPAnnualReimbursementAmt_perProvider,5160.864729,5271.108559,2357.120652,2669.241053,0.000000,0.000000,96000.000000,65800.000000,110.243830
5,count_ClaimID_perProviderOtherPhysician,356.160133,265.884713,422.440361,326.364341,0.000000,0.000000,1872.000000,1234.000000,90.275421
6,count_ClaimID_perProviderClmDiagnosisCode_4,256.087557,198.330248,279.705673,233.094922,0.000000,0.000000,1119.000000,904.000000,57.757308
7,count_ClaimID_perProviderOperatingPhysician,182.315101,138.226010,201.834851,162.340525,0.000000,0.000000,833.000000,638.000000,44.089091
8,mean_InscClaimAmtReimbursed_perAttendingPhysician,936.272500,978.750369,2656.029972,2779.921369,0.000000,0.000000,125000.000000,70000.000000,42.477869
9,mean_InscClaimAmtReimbursed_perProvider,939.383372,981.307906,1427.134188,1654.221667,0.000000,0.000000,57000.000000,57000.000000,41.924534



DISTRIBUTION CHECK COMPLETED
Features checked: 62
Features with identical mean: 1
Features with mean difference > 1: 25


In [9]:
# ============================================================
# FINAL FEATURE LOGIC CHECK
# VERIFY PROVIDER FEATURES DIRECTLY FROM RAW UNSEEN DATA
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("FINAL PROVIDER FEATURE LOGIC CHECK")
print("=" * 70)

# ------------------------------------------------------------
# Recreate the SAME claim-level base used by production
# ------------------------------------------------------------

unseen_claims = pd.concat(
    [
        inpatient_test,
        outpatient_test
    ],
    ignore_index=True
)

unseen_claims = unseen_claims.merge(
    beneficiary_test,
    on="BeneID",
    how="left"
)

unseen_claims = unseen_claims.reset_index(drop=True)

print("Raw unseen claim rows:", len(unseen_claims))
print("Production feature rows:", len(X_production_test))

# ------------------------------------------------------------
# 1. PROVIDER CLAIM COUNT
# ------------------------------------------------------------

expected_provider_count = (
    unseen_claims
    .groupby("Provider")["ClaimID"]
    .transform("count")
)

production_provider_count = (
    X_production_test["count_ClaimID_perProvider"]
)

print("\nProvider claim count")

print(
    "Maximum difference:",
    np.abs(
        expected_provider_count.to_numpy()
        - production_provider_count.to_numpy()
    ).max()
)

print(
    "Different values:",
    (
        expected_provider_count.to_numpy()
        != production_provider_count.to_numpy()
    ).sum()
)

# ------------------------------------------------------------
# 2. PROVIDER + BENEFICIARY COUNT
# ------------------------------------------------------------

expected_provider_bene = (
    unseen_claims
    .groupby("Provider")["BeneID"]
    .transform("nunique")
)

production_provider_bene = (
    X_production_test["count_ClaimID_perProviderBeneID"]
)

print("\nProvider unique beneficiary count")

print(
    "Maximum difference:",
    np.abs(
        expected_provider_bene.to_numpy()
        - production_provider_bene.to_numpy()
    ).max()
)

print(
    "Different values:",
    (
        expected_provider_bene.to_numpy()
        != production_provider_bene.to_numpy()
    ).sum()
)

# ------------------------------------------------------------
# 3. PROVIDER + ATTENDING PHYSICIAN
# ------------------------------------------------------------

expected_attending = (
    unseen_claims
    .groupby("Provider")["AttendingPhysician"]
    .transform(
        lambda x: x.notna().sum()
    )
)

production_attending = (
    X_production_test[
        "count_ClaimID_perProviderAttendingPhysician"
    ]
)

print("\nProvider attending physician count")

print(
    "Maximum difference:",
    np.abs(
        expected_attending.to_numpy()
        - production_attending.to_numpy()
    ).max()
)

print(
    "Different values:",
    (
        expected_attending.to_numpy()
        != production_attending.to_numpy()
    ).sum()
)

# ------------------------------------------------------------
# 4. PROVIDER + OTHER PHYSICIAN
# ------------------------------------------------------------

expected_other = (
    unseen_claims
    .groupby("Provider")["OtherPhysician"]
    .transform(
        lambda x: x.notna().sum()
    )
)

production_other = (
    X_production_test[
        "count_ClaimID_perProviderOtherPhysician"
    ]
)

print("\nProvider other physician count")

print(
    "Maximum difference:",
    np.abs(
        expected_other.to_numpy()
        - production_other.to_numpy()
    ).max()
)

print(
    "Different values:",
    (
        expected_other.to_numpy()
        != production_other.to_numpy()
    ).sum()
)

print("\n" + "=" * 70)
print("FEATURE LOGIC CHECK COMPLETED")
print("=" * 70)

FINAL PROVIDER FEATURE LOGIC CHECK
Raw unseen claim rows: 135392
Production feature rows: 135392

Provider claim count
Maximum difference: 0
Different values: 0

Provider unique beneficiary count
Maximum difference: 0
Different values: 0

Provider attending physician count
Maximum difference: 0
Different values: 0

Provider other physician count
Maximum difference: 0
Different values: 0

FEATURE LOGIC CHECK COMPLETED


In [10]:
# ============================================================
# CHECK PROVIDER AGGREGATION RULE
# ============================================================

print("=" * 70)
print("CHECKING PROVIDER AGGREGATION")
print("=" * 70)

print("Production claims:", len(X_production_test))
print("Provider mapping rows:", len(provider_mapping))

print("\nProvider mapping columns:")
print(provider_mapping.columns.tolist())

print("\nModel configuration:")
print(model_config)

CHECKING PROVIDER AGGREGATION
Production claims: 135392
Provider mapping rows: 135392

Provider mapping columns:
['Provider']

Model configuration:
{'model_name': 'CatBoost', 'feature_count': 62, 'threshold': 0.4, 'random_seed': 42, 'depth': 5, 'iterations': 200, 'learning_rate': 0.1, 'test_accuracy': 0.9127, 'test_precision': 0.915, 'test_recall': 0.8868, 'test_f1': 0.9007, 'test_roc_auc': 0.9542, 'test_pr_auc': 0.9584}


In [12]:
# ============================================================
# LOAD THE TRAINED CATBOOST MODEL
# ============================================================

from pathlib import Path
from catboost import CatBoostClassifier

PROJECT_ROOT = Path.cwd().parent

print("=" * 70)
print("SEARCHING FOR SAVED CATBOOST MODEL")
print("=" * 70)

model_files = []

for pattern in ["*.cbm", "*.pkl", "*.joblib"]:
    model_files.extend(
        PROJECT_ROOT.rglob(pattern)
    )

for file in model_files:
    print(file)

print("=" * 70)

SEARCHING FOR SAVED CATBOOST MODEL
c:\Users\suriy\Desktop\healthcare_fraud\models\catboost_model.cbm
c:\Users\suriy\Desktop\healthcare_fraud\experiments\models\v2_62_features\catboost_model.cbm
c:\Users\suriy\Desktop\healthcare_fraud\models\v2_62_features\catboost_model.cbm
c:\Users\suriy\Desktop\healthcare_fraud\data\production_input_59.pkl
c:\Users\suriy\Desktop\healthcare_fraud\data\production_test_claims_59.pkl
c:\Users\suriy\Desktop\healthcare_fraud\models\encoder.pkl
c:\Users\suriy\Desktop\healthcare_fraud\models\feature_columns.pkl
c:\Users\suriy\Desktop\healthcare_fraud\models\preprocessing_metadata.pkl
c:\Users\suriy\Desktop\healthcare_fraud\.venv\Lib\site-packages\joblib\test\data\joblib_0.10.0_pickle_py27_np17.pkl
c:\Users\suriy\Desktop\healthcare_fraud\.venv\Lib\site-packages\joblib\test\data\joblib_0.10.0_pickle_py33_np18.pkl
c:\Users\suriy\Desktop\healthcare_fraud\.venv\Lib\site-packages\joblib\test\data\joblib_0.10.0_pickle_py34_np19.pkl
c:\Users\suriy\Desktop\healthcare

In [13]:
# ============================================================
# LOAD THE EXACT 62-FEATURE CATBOOST MODEL
# ============================================================

from catboost import CatBoostClassifier
from pathlib import Path
import pickle

PROJECT_ROOT = Path.cwd().parent

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "v2_62_features"
)

# Load CatBoost model
model = CatBoostClassifier()

model.load_model(
    MODEL_DIR / "catboost_model.cbm"
)

# Load saved feature columns
with open(
    MODEL_DIR / "feature_columns.pkl",
    "rb"
) as f:
    saved_feature_columns = pickle.load(f)

# Load saved model configuration
with open(
    MODEL_DIR / "model_config.pkl",
    "rb"
) as f:
    saved_model_config = pickle.load(f)

print("=" * 70)
print("EXACT 62-FEATURE MODEL LOADED")
print("=" * 70)

print("Model:", type(model).__name__)
print("Feature count:", len(saved_feature_columns))

print("\nModel config:")
print(saved_model_config)

print("=" * 70)

EXACT 62-FEATURE MODEL LOADED
Model: CatBoostClassifier
Feature count: 62

Model config:
{'model_name': 'CatBoost', 'feature_count': 62, 'threshold': 0.4, 'random_seed': 42, 'depth': 5, 'iterations': 200, 'learning_rate': 0.1, 'test_accuracy': 0.9127, 'test_precision': 0.915, 'test_recall': 0.8868, 'test_f1': 0.9007, 'test_roc_auc': 0.9542, 'test_pr_auc': 0.9584}


In [14]:
# ============================================================
# VERIFY MODEL vs PRODUCTION FEATURES
# ============================================================

print("=" * 70)
print("MODEL / FEATURE CHECK")
print("=" * 70)

print(
    "Production feature shape:",
    X_production_test.shape
)

print(
    "Saved feature count:",
    len(saved_feature_columns)
)

print(
    "Same feature names:",
    list(X_production_test.columns)
    == list(saved_feature_columns)
)

print("=" * 70)

MODEL / FEATURE CHECK
Production feature shape: (135392, 62)
Saved feature count: 62
Same feature names: True


In [15]:
# ============================================================
# FINAL PROVIDER-LEVEL PREDICTION
# ============================================================

fraud_probability = model.predict_proba(
    X_production_test
)[:, 1]

claim_results = pd.DataFrame({
    "Provider": provider_mapping["Provider"].values,
    "Fraud_Probability": fraud_probability
})

provider_results = (
    claim_results
    .groupby("Provider", as_index=False)
    .agg(
        Claim_Count=("Fraud_Probability", "size"),
        Fraud_Probability=("Fraud_Probability", "mean")
    )
)

threshold = saved_model_config.get(
    "threshold",
    0.4
)

provider_results["Fraud_Prediction"] = (
    provider_results["Fraud_Probability"] >= threshold
).astype(int)

provider_results["Fraud_Status"] = (
    provider_results["Fraud_Prediction"]
    .map({
        0: "Not Fraud",
        1: "Fraud"
    })
)

print("=" * 70)
print("FINAL UNSEEN TEST PROVIDER RESULTS")
print("=" * 70)

print(
    "Total claims:",
    len(claim_results)
)

print(
    "Total providers:",
    len(provider_results)
)

print(
    "Predicted fraud providers:",
    provider_results["Fraud_Prediction"].sum()
)

print(
    "Predicted not-fraud providers:",
    (
        provider_results["Fraud_Prediction"] == 0
    ).sum()
)

print(
    "Predicted fraud provider percentage:",
    round(
        provider_results["Fraud_Prediction"].mean() * 100,
        2
    ),
    "%"
)

print("=" * 70)

FINAL UNSEEN TEST PROVIDER RESULTS
Total claims: 135392
Total providers: 1353
Predicted fraud providers: 94
Predicted not-fraud providers: 1259
Predicted fraud provider percentage: 6.95 %


In [16]:
# ============================================================
# SAVE FINAL UNSEEN TEST PROVIDER RESULTS
# ============================================================

from pathlib import Path

OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_OUTPUT = (
    OUTPUT_DIR
    / "unseen_test_provider_predictions.csv"
)

provider_results.to_csv(
    FINAL_OUTPUT,
    index=False
)

print("=" * 70)
print("FINAL RESULTS SAVED")
print("=" * 70)

print(FINAL_OUTPUT)
print("Rows:", len(provider_results))
print("=" * 70)

FINAL RESULTS SAVED
c:\Users\suriy\Desktop\healthcare_fraud\outputs\unseen_test_provider_predictions.csv
Rows: 1353


In [17]:
# ============================================================
# SAVE CLAIM-LEVEL UNSEEN PREDICTIONS
# ============================================================

claim_results["Fraud_Prediction"] = (
    claim_results["Fraud_Probability"] >= threshold
).astype(int)

claim_results["Fraud_Status"] = (
    claim_results["Fraud_Prediction"]
    .map({
        0: "Not Fraud",
        1: "Fraud"
    })
)

CLAIM_OUTPUT = (
    OUTPUT_DIR
    / "unseen_test_claim_predictions.csv"
)

claim_results.to_csv(
    CLAIM_OUTPUT,
    index=False
)

print("=" * 70)
print("CLAIM-LEVEL RESULTS SAVED")
print("=" * 70)

print(CLAIM_OUTPUT)
print("Rows:", len(claim_results))

CLAIM-LEVEL RESULTS SAVED
c:\Users\suriy\Desktop\healthcare_fraud\outputs\unseen_test_claim_predictions.csv
Rows: 135392


In [18]:
print(provider_results.head(20))
print()
print(provider_results["Fraud_Status"].value_counts())

    Provider  Claim_Count  Fraud_Probability  Fraud_Prediction Fraud_Status
0   PRV51002          205           0.001895                 0    Not Fraud
1   PRV51006          102           0.009228                 0    Not Fraud
2   PRV51009           39           0.021263                 0    Not Fraud
3   PRV51010           38           0.022202                 0    Not Fraud
4   PRV51018          190           0.004161                 0    Not Fraud
5   PRV51019            6           0.000734                 0    Not Fraud
6   PRV51020           45           0.061046                 0    Not Fraud
7   PRV51022          124           0.066368                 0    Not Fraud
8   PRV51028            9           0.000349                 0    Not Fraud
9   PRV51033           41           0.005465                 0    Not Fraud
10  PRV51034           48           0.002026                 0    Not Fraud
11  PRV51039          225           0.066347                 0    Not Fraud
12  PRV51050

In [20]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from data_ingestion import load_from_folder

In [21]:
TEST_FOLDER = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "test"
)

datasets = load_from_folder(
    TEST_FOLDER
)

print("\n" + "=" * 70)
print("DATASET IDENTIFICATION RESULT")
print("=" * 70)

for name, df in datasets.items():

    if name == "_temp_directory":
        continue

    print(
        f"{name:15} : {df.shape}"
    )

Identified beneficiary: Test_Beneficiarydata-1542969243754.csv (63968, 25)
Identified inpatient: Test_Inpatientdata-1542969243754.csv (9551, 30)
Identified outpatient: Test_Outpatientdata-1542969243754.csv (125841, 27)

DATASET IDENTIFICATION RESULT
beneficiary     : (63968, 25)
inpatient       : (9551, 30)
outpatient      : (125841, 27)


In [22]:
beneficiary_test = datasets["beneficiary"]
inpatient_test = datasets["inpatient"]
outpatient_test = datasets["outpatient"]

print(
    beneficiary_test.shape,
    inpatient_test.shape,
    outpatient_test.shape
)

(63968, 25) (9551, 30) (125841, 27)


In [23]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from data_ingestion import load_from_folder, load_from_zip

print("Data ingestion imported successfully.")

Data ingestion imported successfully.


In [24]:
ZIP_PATH = PROJECT_ROOT / "data" / "raw" / "testing.zip"

datasets = load_from_zip(
    ZIP_PATH
)

print("=" * 70)
print("ZIP DATASET IDENTIFICATION RESULT")
print("=" * 70)

for name, df in datasets.items():

    if name == "_temp_directory":
        continue

    print(
        f"{name:15} : {df.shape}"
    )

Identified beneficiary: Test_Beneficiarydata-1542969243754.csv (63968, 25)
Identified inpatient: Test_Inpatientdata-1542969243754.csv (9551, 30)
Identified outpatient: Test_Outpatientdata-1542969243754.csv (125841, 27)
ZIP DATASET IDENTIFICATION RESULT
beneficiary     : (63968, 25)
inpatient       : (9551, 30)
outpatient      : (125841, 27)


In [25]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from prediction_pipeline import predict_from_folder

In [26]:
claim_results, provider_results = (
    predict_from_folder(
        PROJECT_ROOT
        / "data"
        / "raw"
        / "test"
    )
)

Identified beneficiary: Test_Beneficiarydata-1542969243754.csv (63968, 25)
Identified inpatient: Test_Inpatientdata-1542969243754.csv (9551, 30)
Identified outpatient: Test_Outpatientdata-1542969243754.csv (125841, 27)
FINAL 62-FEATURE PREDICTION PREPROCESSING
Saved feature count: 62
Model threshold: 0.4
Merged data shape: (135392, 54)
Final prediction shape: (135392, 62)
Expected feature count: 62
Missing values: 0
62-FEATURE PREPROCESSING PASSED
PREDICTION COMPLETED
Total claims: 135392
Total providers: 1353
Predicted fraud providers: 94
Predicted not-fraud providers: 1259
Fraud provider percentage: 6.95 %


In [16]:
# ============================================================
# RECREATE DEVELOPMENT TRAIN / TEST PROVIDER SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

provider_labels_dev = (
    provider_labels_dev[
        ["Provider", "PotentialFraud"]
    ]
    .drop_duplicates("Provider")
    .copy()
)

dev_train_providers, dev_test_providers = train_test_split(
    provider_labels_dev["Provider"],
    test_size=0.20,
    random_state=42,
    stratify=provider_labels_dev["PotentialFraud"]
)

dev_test_providers = set(dev_test_providers)

print("=" * 70)
print("DEVELOPMENT TEST PROVIDERS")
print("=" * 70)

print(
    "Development test providers:",
    len(dev_test_providers)
)

DEVELOPMENT TEST PROVIDERS
Development test providers: 1082


In [17]:
# ============================================================
# RECREATE DEVELOPMENT TEST CLAIMS
# ============================================================

dev_inpatient_test = inpatient_dev[
    inpatient_dev["Provider"].isin(
        dev_test_providers
    )
].copy()

dev_outpatient_test = outpatient_dev[
    outpatient_dev["Provider"].isin(
        dev_test_providers
    )
].copy()

dev_test_inout = pd.concat(
    [
        dev_inpatient_test,
        dev_outpatient_test
    ],
    ignore_index=True
)

dev_beneficiary_ids = set(
    dev_test_inout["BeneID"].dropna()
)

dev_beneficiary_test = beneficiary_dev[
    beneficiary_dev["BeneID"].isin(
        dev_beneficiary_ids
    )
].copy()

dev_test_claims = dev_test_inout.merge(
    dev_beneficiary_test,
    on="BeneID",
    how="left"
)

print("=" * 70)
print("RECREATED DEVELOPMENT TEST CLAIMS")
print("=" * 70)

print(
    "Inpatient   :",
    dev_inpatient_test.shape
)

print(
    "Outpatient  :",
    dev_outpatient_test.shape
)

print(
    "Beneficiary :",
    dev_beneficiary_test.shape
)

print(
    "Test claims :",
    dev_test_claims.shape
)

RECREATED DEVELOPMENT TEST CLAIMS
Inpatient   : (8978, 30)
Outpatient  : (124798, 27)
Beneficiary : (61836, 25)
Test claims : (133776, 54)


In [18]:
# ============================================================
# PROVIDER-LEVEL EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 70)
print("PROVIDER-LEVEL EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# Confirm row alignment
# ------------------------------------------------------------

if len(dev_test_claims) != len(X_original):
    raise ValueError(
        f"Row mismatch: "
        f"dev_test_claims={len(dev_test_claims)}, "
        f"X_original={len(X_original)}"
    )

# ------------------------------------------------------------
# Claim-level results
# ------------------------------------------------------------

claim_results = pd.DataFrame({
    "Provider":
        dev_test_claims["Provider"]
        .reset_index(drop=True),

    "Fraud_Probability":
        original_probability,

    "Actual_Fraud":
        y_test_original.reset_index(drop=True)
})

# ------------------------------------------------------------
# Aggregate to PROVIDER
# ------------------------------------------------------------

provider_results = (
    claim_results
    .groupby("Provider")
    .agg(
        Claim_Count=(
            "Provider",
            "size"
        ),

        Fraud_Probability=(
            "Fraud_Probability",
            "mean"
        ),

        Actual_Fraud=(
            "Actual_Fraud",
            "first"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# Provider prediction
# ------------------------------------------------------------

provider_results["Fraud_Prediction"] = (
    provider_results["Fraud_Probability"] >= 0.4
).astype(int)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

y_provider = provider_results["Actual_Fraud"]
p_provider = provider_results["Fraud_Prediction"]

print(
    "Number of providers:",
    len(provider_results)
)

print("\nActual provider distribution:")
print(
    y_provider.value_counts()
    .sort_index()
)

print("\nPredicted provider distribution:")
print(
    p_provider.value_counts()
    .sort_index()
)

print("\nProvider-level metrics:")

print(
    "Accuracy :",
    round(
        accuracy_score(
            y_provider,
            p_provider
        ),
        4
    )
)

print(
    "Precision:",
    round(
        precision_score(
            y_provider,
            p_provider,
            zero_division=0
        ),
        4
    )
)

print(
    "Recall   :",
    round(
        recall_score(
            y_provider,
            p_provider,
            zero_division=0
        ),
        4
    )
)

print(
    "F1       :",
    round(
        f1_score(
            y_provider,
            p_provider,
            zero_division=0
        ),
        4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_provider,
        p_provider
    )
)

print("\nSample results:")
display(
    provider_results.head(20)
)

print("=" * 70)

PROVIDER-LEVEL EVALUATION
Number of providers: 1082

Actual provider distribution:
Actual_Fraud
0    981
1    101
Name: count, dtype: int64

Predicted provider distribution:
Fraud_Prediction
0    988
1     94
Name: count, dtype: int64

Provider-level metrics:
Accuracy : 0.9436
Precision: 0.7128
Recall   : 0.6634
F1       : 0.6872

Confusion Matrix:
[[954  27]
 [ 34  67]]

Sample results:


,Provider,Claim_Count,Fraud_Probability,Actual_Fraud,Fraud_Prediction
0,PRV51005,1165,0.005734,1,0
1,PRV51008,43,0.010218,0,0
2,PRV51011,58,0.000324,0,0
3,PRV51012,48,0.000667,0,0
4,PRV51016,6,0.000345,0,0
5,PRV51017,515,0.026453,0,0
6,PRV51029,93,0.000709,0,0
7,PRV51041,34,0.002599,0,0
8,PRV51054,57,0.002957,0,0
9,PRV51059,96,0.920519,1,1


In [19]:
# ============================================================
# FINAL UNSEEN TEST - PROVIDER LEVEL PREDICTIONS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 70)
print("FINAL UNSEEN TEST - PROVIDER LEVEL RESULTS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Generate claim-level fraud probabilities
# ------------------------------------------------------------

fraud_probability = model.predict_proba(
    X_production_test
)[:, 1]

threshold = model_config["threshold"]

claim_prediction = (
    fraud_probability >= threshold
).astype(int)

print("Total claim rows:", len(X_production_test))
print("Threshold:", threshold)

# ------------------------------------------------------------
# 2. Get Provider ID for every production claim
# ------------------------------------------------------------

# provider_mapping should contain the Provider corresponding
# to the production feature rows.

if isinstance(provider_mapping, pd.DataFrame):

    if "Provider" in provider_mapping.columns:
        providers = (
            provider_mapping["Provider"]
            .reset_index(drop=True)
        )

    else:
        raise ValueError(
            "provider_mapping DataFrame does not contain "
            "'Provider' column."
        )

else:

    providers = pd.Series(
        provider_mapping,
        name="Provider"
    ).reset_index(drop=True)


# ------------------------------------------------------------
# 3. Safety check
# ------------------------------------------------------------

if len(providers) != len(X_production_test):

    raise ValueError(
        "Provider mapping length does not match "
        "production prediction rows.\n"
        f"Providers       : {len(providers)}\n"
        f"Predictions     : {len(X_production_test)}"
    )

# ------------------------------------------------------------
# 4. Create claim-level prediction dataframe
# ------------------------------------------------------------

claim_predictions = pd.DataFrame({

    "Provider": providers,

    "Fraud_Probability": fraud_probability,

    "Claim_Fraud_Prediction": claim_prediction

})


# ------------------------------------------------------------
# 5. Aggregate claims → PROVIDER
# ------------------------------------------------------------

provider_predictions = (
    claim_predictions
    .groupby("Provider")
    .agg(

        Claim_Count=(
            "Provider",
            "size"
        ),

        Fraud_Probability=(
            "Fraud_Probability",
            "mean"
        ),

        Fraud_Claims=(
            "Claim_Fraud_Prediction",
            "sum"
        )

    )
    .reset_index()
)


# ------------------------------------------------------------
# 6. Final PROVIDER fraud decision
# ------------------------------------------------------------

provider_predictions["Fraud_Prediction"] = (
    provider_predictions["Fraud_Probability"]
    >= threshold
).astype(int)


# ------------------------------------------------------------
# 7. Convert prediction to readable label
# ------------------------------------------------------------

provider_predictions["Fraud_Status"] = (
    provider_predictions["Fraud_Prediction"]
    .map({
        0: "Not Fraud",
        1: "Fraud"
    })
)


# ------------------------------------------------------------
# 8. Sort highest-risk providers first
# ------------------------------------------------------------

provider_predictions = (
    provider_predictions
    .sort_values(
        "Fraud_Probability",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. Save final provider-level results
# ------------------------------------------------------------

OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_PATH = (
    OUTPUT_DIR
    / "unseen_test_provider_predictions.csv"
)

provider_predictions.to_csv(
    OUTPUT_PATH,
    index=False
)


# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL PROVIDER-LEVEL RESULTS")
print("=" * 70)

print(
    "Total claims evaluated :",
    len(claim_predictions)
)

print(
    "Total providers        :",
    len(provider_predictions)
)

print(
    "Predicted Fraud providers:",
    (
        provider_predictions["Fraud_Prediction"]
        == 1
    ).sum()
)

print(
    "Predicted Not-Fraud providers:",
    (
        provider_predictions["Fraud_Prediction"]
        == 0
    ).sum()
)

print(
    "\nFraud provider percentage:",
    round(
        provider_predictions["Fraud_Prediction"]
        .mean() * 100,
        2
    ),
    "%"
)

print(
    "\nSaved to:"
)

print(OUTPUT_PATH)

print("\nTop 20 highest-risk providers:")

display(
    provider_predictions.head(20)
)

print("\n" + "=" * 70)
print("FINAL UNSEEN TEST PREDICTION COMPLETED")
print("=" * 70)

FINAL UNSEEN TEST - PROVIDER LEVEL RESULTS
Total claim rows: 135392
Threshold: 0.4

FINAL PROVIDER-LEVEL RESULTS
Total claims evaluated : 135392
Total providers        : 1353
Predicted Fraud providers: 663
Predicted Not-Fraud providers: 690

Fraud provider percentage: 49.0 %

Saved to:
C:\Users\suriy\Desktop\healthcare_fraud\outputs\unseen_test_provider_predictions.csv

Top 20 highest-risk providers:


,Provider,Claim_Count,Fraud_Probability,Fraud_Claims,Fraud_Prediction,Fraud_Status
0,PRV53395,28,0.996001,28,1,Fraud
1,PRV56534,72,0.986004,72,1,Fraud
2,PRV55693,23,0.984609,23,1,Fraud
3,PRV54406,24,0.982275,24,1,Fraud
4,PRV55328,84,0.982225,84,1,Fraud
5,PRV53250,86,0.981040,86,1,Fraud
6,PRV57229,205,0.978966,205,1,Fraud
7,PRV54650,1658,0.977463,1641,1,Fraud
8,PRV51107,14,0.976226,14,1,Fraud
9,PRV54850,71,0.975199,71,1,Fraud



FINAL UNSEEN TEST PREDICTION COMPLETED


In [20]:
# ============================================================
# CHECK PROVIDER FRAUD PROBABILITY DISTRIBUTION
# ============================================================

print("=" * 70)
print("UNSEEN TEST PROVIDER PROBABILITY DISTRIBUTION")
print("=" * 70)

print(
    provider_predictions["Fraud_Probability"].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print("\nProbability ranges:")

bins = [
    0,
    0.01,
    0.05,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    1.00
]

print(
    pd.cut(
        provider_predictions["Fraud_Probability"],
        bins=bins,
        include_lowest=True
    ).value_counts()
    .sort_index()
)

print("=" * 70)

UNSEEN TEST PROVIDER PROBABILITY DISTRIBUTION
count    1353.000000
mean        0.420456
std         0.351729
min         0.000004
1%          0.000259
5%          0.001386
10%         0.004700
25%         0.049781
50%         0.380885
75%         0.774179
90%         0.911360
95%         0.939587
99%         0.971035
max         0.996001
Name: Fraud_Probability, dtype: float64

Probability ranges:
Fraud_Probability
(-0.001, 0.01]    196
(0.01, 0.05]      143
(0.05, 0.1]        85
(0.1, 0.2]        118
(0.2, 0.3]         75
(0.3, 0.4]         73
(0.4, 0.5]         69
(0.5, 0.6]         75
(0.6, 0.7]        102
(0.7, 0.8]        111
(0.8, 0.9]        151
(0.9, 1.0]        155
Name: count, dtype: int64


In [21]:
# ============================================================
# COMPARE DEVELOPMENT vs UNSEEN PROVIDER PROBABILITIES
# ============================================================

print("=" * 70)
print("DEVELOPMENT vs UNSEEN PROVIDER PROBABILITIES")
print("=" * 70)

print("\nDEVELOPMENT TEST")
print(
    provider_results["Fraud_Probability"].describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print("\nOFFICIAL UNSEEN TEST")
print(
    provider_predictions["Fraud_Probability"].describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print("=" * 70)

DEVELOPMENT vs UNSEEN PROVIDER PROBABILITIES

DEVELOPMENT TEST
count    1082.000000
mean        0.092872
std         0.237284
min         0.000005
10%         0.000345
25%         0.001066
50%         0.003816
75%         0.024316
90%         0.328052
95%         0.830603
99%         0.999863
max         0.999975
Name: Fraud_Probability, dtype: float64

OFFICIAL UNSEEN TEST
count    1353.000000
mean        0.420456
std         0.351729
min         0.000004
10%         0.004700
25%         0.049781
50%         0.380885
75%         0.774179
90%         0.911360
95%         0.939587
99%         0.971035
max         0.996001
Name: Fraud_Probability, dtype: float64
